# Lab 08 — Working with a Live Public API
**Data Outside Files Track** · Intermediate · ~40 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. GET JSON from a public API with urllib and a User-Agent
2. Retry 429/5xx with exponential backoff
3. Cache responses and flatten nested JSON to rows
4. Save api_snapshot.csv; fall back to offline fixture

## Datasets (this folder)
- `offline_fixture.json` — **upload** in Colab (or keep next to the notebook locally)

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-01-foundations-python-math-data/labs/lab-08-live-public-api/lab-08-live-public-api.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0 (bootstrap)** first — it pulls `dataset.zip` from the lab manifest into `/content/ml_lab` (falls back to public raw URLs, then local files).
3. **Runtime → Run all** (GPU not required for Course 1).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 fetches `dataset.zip` from the manifest → `Runtime → Run all`.


### Setup (VLABS bootstrap)

Run the next cell (Cell 0) once. Fetch order: hosted `manifest.json` → `dataset.zip` extracted to `/content/ml_lab/<lab_id>` → per-file public raw URLs → local files next to this notebook. No-op when files already exist.


In [ ]:
# Cell 0 — VLABS bootstrap: run first. Works on Colab (direct-open URL) and locally.
import io, json, os, urllib.request, zipfile

LAB_ID = "lab-08-live-public-api"
# Hosted manifest (matheshcp/ai_course_content, branch main).
MANIFEST_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-01-foundations-python-math-data/bundles/lab-08-live-public-api/manifest.json"
# Alternative: backend proxy to S3 — uncomment to use instead:
# MANIFEST_URL = f"https://api.vlabs.test/colab/{LAB_ID}/manifest"
ON_COLAB = os.path.isdir("/content")
DATA_DIR = f"/content/ml_lab/{LAB_ID}" if ON_COLAB else "."

def _fetch(url, timeout=30):
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return r.read()

def _ensure_file(filename, url=None):
    """Local-first single-file fetch (also used by lesson load cells)."""
    for base in (DATA_DIR, "."):
        p = os.path.join(base, filename)
        if os.path.exists(p):
            print(f"found {p}")
            return p
    if not url:
        raise FileNotFoundError(
            f"{filename} missing: open via Start Lab (bundle) or add it next to the notebook")
    os.makedirs(DATA_DIR, exist_ok=True)
    dest = os.path.join(DATA_DIR, filename)
    print(f"downloading {filename} ...")
    urllib.request.urlretrieve(url, dest)
    print(f"saved {dest}")
    return dest

ensure = _ensure_file  # compat alias for lesson load cells

def vlabs_bootstrap():
    # 1) Hosted manifest -> dataset.zip -> DATA_DIR (direct-open path)
    try:
        m = json.loads(_fetch(MANIFEST_URL).decode("utf-8"))
        dz = m.get("dataset_zip")
        if dz:
            print(f"manifest ok: {MANIFEST_URL}")
            os.makedirs(DATA_DIR, exist_ok=True)
            zpath = os.path.join(DATA_DIR, "dataset.zip")
            urllib.request.urlretrieve(dz, zpath)
            with zipfile.ZipFile(zpath) as z:
                z.extractall(DATA_DIR)
            print(f"extracted dataset.zip -> {DATA_DIR}")
    except Exception as e:
        print(f"manifest skip ({e}); using file fallbacks")
    # 2) Per-file fallbacks (public raw URLs; local files are a no-op hit)
    _ensure_file("offline_fixture.json")
    # 3) Work from the data dir on Colab so relative paths resolve
    if ON_COLAB and DATA_DIR != ".":
        os.chdir(DATA_DIR)
        print(f"cwd -> {DATA_DIR}")

vlabs_bootstrap()


## Data Outside Files Track: HTTP, JSON, Caching, Backoff

> **Scenario:** Fetch a free public JSON API (Open-Meteo forecast — no API key), handle timeouts and 429s with exponential backoff, cache the response to `api_cache.json`, flatten nested JSON into a DataFrame, and save `api_snapshot.csv`. An **offline fixture** is shipped so the lab runs with no network.
>
> **You will learn:** `urllib.request`, JSON parsing, status codes, caching, schema flattening, polite retry.
> **Time:** ~40 minutes. **Level:** Intermediate. **Needs:** Python 3.8+ (pandas optional). **Env:** 🟢 Colab only.

### API mental map

| Web concept | Python | Failure mode |
|---|---|---|
| GET URL | `urllib.request.urlopen` | timeout / DNS |
| JSON body | `json.loads` | decode error |
| 200 / 429 / 5xx | check `status` | rate limit |
| Retry | sleep `2**attempt` | thundering herd |
| Cache | write local JSON | stale data |
| Nested → table | dict flatten | missing keys |

---

### 1. Fetch with retry / backoff (fixture fallback)

In [ ]:
import json, os, time, urllib.error, urllib.request

API_URL = (
    "https://api.open-meteo.com/v1/forecast"
    "?latitude=-37.81&longitude=144.96"
    "&daily=temperature_2m_max,temperature_2m_min,precipitation_probability_max"
    "&timezone=auto&forecast_days=3"
)
FIXTURE = "offline_fixture.json"
CACHE = "api_cache.json"

def fetch_json(url, cache_path=CACHE, fixture=FIXTURE,
               timeout=10, max_retries=4, use_network=True):
    """GET JSON with exponential backoff. Falls back to local fixture."""
    if os.path.exists(cache_path) and use_network is not None:
        # Prefer fresh cache only if caller allows; default: try network first below
        pass
    if not use_network:
        with open(fixture, encoding="utf-8") as f:
            return json.load(f), "fixture"
    last_err = None
    for attempt in range(max_retries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "ailab-lab08/1.0"})
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                if resp.status == 429:
                    raise urllib.error.HTTPError(url, 429, "Too Many Requests", resp.headers, None)
                data = json.loads(resp.read().decode("utf-8"))
            with open(cache_path, "w", encoding="utf-8") as f:
                json.dump(data, f, indent=2)
            return data, "network"
        except urllib.error.HTTPError as e:
            last_err = e
            if e.code == 429 or e.code >= 500:
                time.sleep(2 ** attempt)  # 1s, 2s, 4s, 8s
                continue
            break  # 4xx other than 429: don't retry
        except (urllib.error.URLError, TimeoutError, OSError) as e:
            last_err = e
            time.sleep(2 ** attempt)
    print(f"network failed ({last_err}); using fixture")
    with open(fixture, encoding="utf-8") as f:
        return json.load(f), "fixture"

# Try network; if you're offline or Colab blocks egress, fixture kicks in.
payload, source = fetch_json(API_URL, use_network=True)
print("source:", source)  # network | fixture
print(list(payload.keys()))


> Why backoff? Hammering a 429 endpoint gets your IP banned. `2**attempt` spreads retries out; always set a User-Agent.

---

### 2. Inspect and flatten nested JSON

In [ ]:
print("timezone:", payload.get("timezone"))
daily = payload["daily"]          # nested object
times = daily["time"]             # parallel arrays
print("days:", times)

# Flatten: one dict per day
records = []
for i, day in enumerate(times):
    records.append({
        "date": day,
        "tmax": daily["temperature_2m_max"][i],
        "tmin": daily["temperature_2m_min"][i],
        "precip_prob": daily["precipitation_probability_max"][i],
        "source": source,
    })
print(records[0])
# e.g. {'date': '2026-09-22', 'tmax': 15.4, 'tmin': 8.2, 'precip_prob': 20, 'source': 'fixture'}


Mental model: API gave you **column-oriented** JSON; analysts want **row-oriented** records.

---

### 3. DataFrame (or list) + save CSV

In [ ]:
try:
    import pandas as pd
    df = pd.DataFrame(records)
    print(df)
    df.to_csv("api_snapshot.csv", index=False)
except ImportError:
    import csv
    with open("api_snapshot.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(records[0].keys()))
        w.writeheader(); w.writerows(records)
    print("wrote api_snapshot.csv (stdlib)")

print("saved api_snapshot.csv rows:", len(records))  # 3


---

### 4. Cache behaviour

In [ ]:
# Second call with use_network=False reads the fixture (or your cache) — no HTTP.
payload2, src2 = fetch_json(API_URL, use_network=False)
print("second source:", src2)  # fixture
print("fixture keys:", sorted(payload2.keys()))
assert "daily" in payload2, "fixture must expose a daily block"

# If you saved a real network response, inspect the cache file:
if os.path.exists(CACHE):
    with open(CACHE, encoding="utf-8") as f:
        print("cache bytes:", len(f.read()))


---

## Exercises (do these!)

### Exercise 1 — Retry with exponential backoff
Simulate three 429 responses then success: write a loop that sleeps `2**attempt` seconds (use `0.01` as a stand-in sleep in tests) and returns a payload after the 4th try. Print the sleep schedule.
*Expected sleeps: 1, 2, 4 (then success on attempt 3 → return). With `max_retries=4`: sequence of waits 1,2,4,8 if all fail.*

<details>
<summary>Hint</summary>

```text
max_retries = 4
for attempt in range(max_retries):
    try:
        ...  # request
        break
    except RateLimited:
        time.sleep(2 ** attempt)
```

</details>

### Exercise 2 — Flatten nested JSON to columns
Given `payload["daily"]` with parallel arrays, build a list of row dicts (date, tmax, tmin, precip_prob). Print `len` and the first row.
*Expected: 3 rows (forecast_days=3); first row keys `date, tmax, tmin, precip_prob`.*

<details>
<summary>Hint</summary>

`for i, day in enumerate(times)` then index each list with `i`.
</details>

### Exercise 3 — Save `api_snapshot.csv`
Write the flattened records to CSV (pandas or `csv` module). Confirm the file exists and row count == number of days.
*Expected: `api_snapshot.csv` with 1 header + 3 data rows.*

<details>
<summary>Hint</summary>

`df.to_csv("api_snapshot.csv", index=False)` or `csv.DictWriter`.
</details>

---

## Solutions

In [ ]:
# --- Solution 1 ---
import time
def retry_demo(max_retries=4, sleep=time.sleep):
    sleeps = []
    for attempt in range(max_retries):
        # pretend attempt < 2 raises 429
        if attempt < 2:
            w = 2 ** attempt
            sleeps.append(w)
            sleep(w)
            continue
        return {"ok": True, "attempt": attempt}, sleeps
    return None, sleeps
# Use real time.sleep in production; in tests inject sleep=lambda s: None.
data, schedule = retry_demo(sleep=lambda s: None)
print("sleeps:", schedule, "-> success at", data["attempt"])
# sleeps: [1, 2] -> success at 2

# --- Solution 2 ---
daily = payload["daily"]
rows = [
    {"date": t,
     "tmax": daily["temperature_2m_max"][i],
     "tmin": daily["temperature_2m_min"][i],
     "precip_prob": daily["precipitation_probability_max"][i]}
    for i, t in enumerate(daily["time"])
]
print(len(rows), rows[0])  # 3 {date: ..., tmax: ..., ...}

# --- Solution 3 ---
# (from Section 3) after writing:
import os
assert os.path.exists("api_snapshot.csv")
with open("api_snapshot.csv", encoding="utf-8") as f:
    n_lines = sum(1 for _ in f)
print("lines including header:", n_lines)  # 4


### What to learn next
- `requests` library (`response.raise_for_status()`, sessions).
- OpenAPI/Swagger docs; API keys via environment variables — never commit secrets.
- Pagination + `asyncio` for many endpoints.
- Cheer sheet: GET → check status → backoff on 429/5xx → cache → flatten → CSV.

*Files in this folder: `offline_fixture.json` (always works offline) · outputs `api_cache.json`, `api_snapshot.csv`.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
